# Teachable Machine nachgebaut – Variante kNN

**Ziel:** Verstehen, was Teachable Machine *im Kern* tut – ohne grafische Oberfläche, in wenigen Zellen.

Diese Version bildet das **ursprüngliche** Teachable Machine (2017, Webcam-Experiment) nach. Es beruht auf zwei Bausteinen:

1. **MobileNetV2** als *eingefrorener Merkmals-Extraktor*. Ein großes, auf 1,4 Mio. Bildern (ImageNet) vortrainiertes Netz. Wir trainieren es **nicht** – wir nutzen nur seinen „Blick": Es verwandelt jedes Bild in einen Zahlenvektor mit 1280 Werten (ein *Embedding*), der das Wesentliche des Bildes beschreibt.
2. **k-Nearest-Neighbors (kNN)** als Klassifikator. Kein Training, keine Gewichte – nur: *„Welchen bekannten Bildern ähnelt das neue Bild am meisten?"*

> **Didaktischer Kern:** Das eigentliche „Lernen" passiert hier gar nicht. Wir berechnen einmal die Embeddings und vergleichen dann nur noch Abstände. Genau das macht den Reiz – und die Grenzen – dieser Variante aus.

*(Das heutige Teachable Machine ersetzt den kNN-Schritt durch einen kleinen trainierten Dense-Kopf. Das bauen wir in einem späteren Notebook.)*


## 1 | Bibliotheken

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torchvision.models import MobileNet_V2_Weights
from torch.hub import load_state_dict_from_url
from torch.utils.data import DataLoader

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

import matplotlib.pyplot as plt
from PIL import Image

# Hardware-Erkennung (CUDA für Nvidia, MPS für Apple Silicon, sonst CPU)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Aktuelles Gerät: {DEVICE}")

## 2 | Konfiguration

Alle Stellschrauben an einem Ort. Die Bilder liegen bereits als **224×224 RGB** vor – genau das Format, das MobileNetV2 erwartet.

In [ ]:
TRAIN_DIR = "Datasets/Train"   # enthält Unterordner KlasseA, KlasseB, ...
TEST_DIR  = "Datasets/Test"
MODELS_DIR = "Models"          # hierhin werden die MobileNet-Gewichte (.pth) gelegt

K = 3            # Anzahl der Nachbarn beim kNN
BATCH_SIZE = 16  # nur für die Embedding-Berechnung relevant

## 3 | Bilder laden

`ImageFolder` liest die Ordnerstruktur direkt ein: **Jeder Unterordner ist eine Klasse.** Die Reihenfolge der Klassennamen (alphabetisch) bestimmt die Label-Nummern.

Wichtig: Die Bilder sind schon 224×224. Wir wandeln sie nur in Tensoren um und **normalisieren** sie so, wie MobileNetV2 es im Training „gewohnt" war (ImageNet-Mittelwerte). Ohne diese Normalisierung sieht das Netz die Bilder „falsch belichtet" und liefert schlechtere Embeddings.

In [ ]:
preprocess = transforms.Compose([
    transforms.ToTensor(),                      # PIL-Bild -> Tensor, Werte 0..1
    transforms.Normalize(                       # so wie ImageNet trainiert wurde
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=preprocess)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=preprocess)

classes = train_ds.classes
print("Klassen:", classes)
print("Trainingsbilder:", len(train_ds), "| Testbilder:", len(test_ds))

**Ein kurzer Blick in die Daten** – wir zeigen je ein Beispielbild pro Klasse:

In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(3*len(classes), 3))
if len(classes) == 1: axes = [axes]
for ax, cls in zip(axes, classes):
    # erstes Bild dieser Klasse über den gespeicherten Dateipfad laden (unnormalisiert, zum Anschauen)
    path = next(p for p, lbl in train_ds.samples if classes[lbl] == cls)
    ax.imshow(Image.open(path)); ax.set_title(cls); ax.axis("off")
plt.tight_layout(); plt.show()

## 4 | MobileNetV2 als eingefrorener Merkmals-Extraktor

Wir laden das vortrainierte Netz und **schneiden seinen Klassifikationskopf ab** (`classifier = Identity`). Übrig bleibt der Teil, der ein Bild in einen 1280-Werte-Vektor übersetzt.

**Wo liegen die Gewichte?** Standardmäßig würde torchvision die `.pth`-Datei in einen versteckten System-Cache legen. Wir lenken sie stattdessen ins lokale Unterverzeichnis `Models/` um (`load_state_dict_from_url(..., model_dir=MODELS_DIR)`):
- **Erster Lauf:** Datei wird nach `Models/mobilenet_v2-...pth` heruntergeladen (ca. 14 MB, braucht Internet).
- **Jeder weitere Lauf:** Die Datei ist da und wird direkt von der Platte gelesen – **kein** erneuter Download, kein Internet nötig.

Zwei weitere Dinge sind entscheidend:
- **`eval()`** schaltet Trainings-Spezialitäten (z. B. BatchNorm-Updates) ab.
- **`requires_grad = False`** friert alle Gewichte ein – wir trainieren hier nichts.

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

# Gewichte nach Models/ herunterladen bzw. von dort lesen
url = MobileNet_V2_Weights.IMAGENET1K_V1.url
state_dict = load_state_dict_from_url(url, model_dir=MODELS_DIR, progress=True)

net = models.mobilenet_v2(weights=None)   # leeres Gerüst ...
net.load_state_dict(state_dict)           # ... mit unseren lokalen Gewichten füllen

net.classifier = nn.Identity()   # 1000-Klassen-Kopf entfernen -> 1280-dim Embedding
net.eval().to(DEVICE)
for p in net.parameters():
    p.requires_grad = False

print("Gewichte aus:", os.path.join(MODELS_DIR, url.split("/")[-1]))
print("MobileNetV2 geladen. Ausgabe pro Bild: 1280 Zahlen (das Embedding).")

## 5 | Embeddings berechnen – der einzige „teure" Schritt

Jetzt schicken wir **jedes** Trainings- und Testbild einmal durch MobileNetV2 und sammeln die 1280-dim Vektoren ein. Danach arbeiten wir nur noch mit diesen Zahlen weiter – die Bilder selbst brauchen wir nicht mehr.

> Das ist der Grund, warum Teachable Machine sich so schnell anfühlt: Diese Vorberechnung passiert einmal, alles Weitere ist nur noch Vektor-Vergleich.

In [ ]:
@torch.no_grad()
def embeddings(dataset):
    X, y = [], []
    for xb, yb in DataLoader(dataset, batch_size=BATCH_SIZE):
        feats = net(xb.to(DEVICE)).cpu().numpy()
        X.append(feats); y.append(yb.numpy())
    return np.concatenate(X), np.concatenate(y)

X_train, y_train = embeddings(train_ds)
X_test,  y_test  = embeddings(test_ds)

print("Train-Embeddings:", X_train.shape)   # (Anzahl Bilder, 1280)
print("Test-Embeddings: ", X_test.shape)

## 6 | kNN von Hand – an *einem* Testbild

Bevor wir eine fertige Bibliothek nehmen, machen wir das Prinzip einmal sichtbar. Für **ein** Testbild:

1. Berechne den Abstand seines Embeddings zu **allen** Trainings-Embeddings (euklidischer Abstand).
2. Nimm die **K nächsten** Nachbarn.
3. Lass sie **abstimmen**: Die häufigste Klasse unter den Nachbarn ist die Vorhersage.

Mehr ist kNN nicht. Kein Training, keine Gewichte – nur Abstände und ein Mehrheitsvotum.

In [ ]:
i = 0   # Index des Testbildes, das wir untersuchen
query = X_test[i]

# 1) Abstand zu allen Trainingsbildern
dists = np.linalg.norm(X_train - query, axis=1)

# 2) die K nächsten Nachbarn
nn_idx = np.argsort(dists)[:K]
nn_classes = [classes[y_train[j]] for j in nn_idx]

# 3) Mehrheitsvotum
vote = max(set(nn_classes), key=nn_classes.count)

print("Wahre Klasse:      ", classes[y_test[i]])
print("Nachbarn (Klassen):", nn_classes)
print("Vorhersage (Votum):", vote)

**Anschaulich:** das Testbild und seine K nächsten Nachbarn aus den Trainingsdaten.

In [ ]:
fig, axes = plt.subplots(1, K+1, figsize=(3*(K+1), 3))
# Testbild (über Dateipfad, unnormalisiert)
axes[0].imshow(Image.open(test_ds.samples[i][0]))
axes[0].set_title(f"NEU\nwahr: {classes[y_test[i]]}"); axes[0].axis("off")
# Nachbarn
for ax, j in zip(axes[1:], nn_idx):
    ax.imshow(Image.open(train_ds.samples[j][0]))
    ax.set_title(f"Nachbar\n{classes[y_train[j]]}\nd={dists[j]:.1f}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 7 | kNN für alle Testbilder – mit scikit-learn

`KNeighborsClassifier` macht **genau dasselbe** wie unsere Handrechnung oben (Standard-Abstand = euklidisch), nur effizient für alle Bilder auf einmal.

- **`fit`** „lernt" nichts – es **merkt sich** nur die Trainings-Embeddings.
- **`predict`** sucht für jedes Testbild die K Nachbarn und stimmt ab.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=K)
knn.fit(X_train, y_train)          # speichert nur die Trainingsdaten

y_pred = knn.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test-Genauigkeit: {acc:.1%}")

## 8 | Auswertung

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel("Vorhersage"); ax.set_ylabel("Wahre Klasse"); ax.set_title("Konfusionsmatrix")
for r in range(len(classes)):
    for c in range(len(classes)):
        ax.text(c, r, cm[r, c], ha="center", va="center",
                color="white" if cm[r, c] > cm.max()/2 else "black")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

**Ein paar Beispielvorhersagen** – richtige in Schwarz, Fehler in Rot:

In [ ]:
n_show = min(8, len(test_ds))
cols = 4; rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
for ax, idx in zip(np.array(axes).ravel(), range(n_show)):
    ax.imshow(Image.open(test_ds.samples[idx][0]))
    wahr, pred = classes[y_test[idx]], classes[y_pred[idx]]
    farbe = "black" if wahr == pred else "red"
    ax.set_title(f"wahr: {wahr}\nPred: {pred}", color=farbe, fontsize=10); ax.axis("off")
for ax in np.array(axes).ravel()[n_show:]: ax.axis("off")
plt.tight_layout(); plt.show()

## 9 | Was wir gelernt haben – und die Brücke zum nächsten Mal

**Das Prinzip:** Ein großes vortrainiertes Netz (MobileNetV2) liefert aussagekräftige Embeddings. Darauf reicht ein einfacher kNN, um neue Bilder zu klassifizieren – *ganz ohne eigenes Training*. Genau so funktionierte das erste Teachable Machine.

**Die Grenzen von kNN** – und warum das heutige Teachable Machine umgestiegen ist:
- Das „Modell" **ist** der gesamte Trainingssatz – es muss alle Embeddings speichern und bei jeder Vorhersage gegen alle vergleichen (langsam, groß, schlecht exportierbar).
- Alle 1280 Dimensionen zählen **gleich** – Ausreißer und falsch gelabelte Bilder schlagen voll durch.
- Die „Konfidenz" ist nur ein Stimmenverhältnis, keine saubere Wahrscheinlichkeit.

**Nächster Schritt:** Wir tauschen den kNN gegen einen kleinen **Dense-Kopf** (`nn.Sequential`), der auf denselben Embeddings *trainiert* wird. Gleiche Features, anderer Klassifikator – und genau hier zeigt sich der Unterschied zwischen *Beispiele merken* und *Muster lernen*.